# Visualizing Pokemon Web Content
Using t-SNE to reduce the dimensionality of ChromaDB embeddings and visualising them.

In [1]:
import os
import sys

# Change directory to the project root so imports and relative paths work
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

import chromadb
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import plotly.express as px

from utils.config import settings

In [8]:
# Connect to ChromaDB
chroma_client = chromadb.PersistentClient(path=str(settings.VECTOR_DB_DIR))
collection = chroma_client.get_collection(name='pokemon_web_content')

# Fetch data
MAXIMUM_DATAPOINTS = 1000
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)

if result.get("embeddings") is None or len(result.get("embeddings")) == 0:
    print("No data found in pokemon_web_content collection. Please ingest web pages first!")
else:
    vectors = np.array(result['embeddings'])
    documents = result['documents']
    
    # We use 'url' as the category since 'category' does not exist in our metadata
    categories = [metadata.get('url', 'Unknown') for metadata in result['metadatas']]
    
    CATEGORIES = list(set(categories))
    COLORS = px.colors.qualitative.Plotly * 50  # Ensure enough colors
    colors = [COLORS[CATEGORIES.index(c)] for c in categories]

InternalError: Error executing plan: Internal error: Error finding id

In [5]:
if result.get("embeddings") is not None and len(result.get("embeddings")) > 0:
    # Let's try a 2D chart
    # TSNE stands for t-distributed Stochastic Neighbor Embedding - it's a common technique for reducing dimensionality of data
    
    # The perplexity metric must be lower than the number of dimensions/vectors
    perplexity = min(30, max(2, len(vectors) - 1))
    
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    reduced_vectors = tsne.fit_transform(vectors)
    
    # Create the 2D scatter plot
    fig = go.Figure(data=[go.Scatter(
        x=reduced_vectors[:, 0],
        y=reduced_vectors[:, 1],
        mode='markers',
        marker=dict(size=8, color=colors, opacity=0.7),
        text=[f"Url: {c}<br>Text: {d[:100]}..." for c, d in zip(categories, documents)],
        hoverinfo='text'
    )])
    
    fig.update_layout(
        title='2D Chroma Vectorstore Visualization (pokemon_web_content)',
        xaxis_title='TSNE Component 1', 
        yaxis_title='TSNE Component 2',
        width=1200,
        height=800,
        margin=dict(r=20, b=10, l=10, t=40)
    )
    
    fig.show()